### Ingestión del archivo language_role.json

In [0]:
%run '../includes/configuration'

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

##### Paso 1. Leer el archivo

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
languages_roles_schema = StructType(fields=[
    StructField("roleId", IntegerType(), False),
    StructField("languageRole", StringType(), True),
])

In [0]:
languages_roles_df = spark.read \
                .schema(languages_roles_schema) \
                .option("multiline",True) \
                .json(f"{bronze_folder_path}/{v_file_date}/language_role.json")

##### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit, trim

In [0]:
languages_roles_with_columns_df = languages_roles_df \
                        .withColumnRenamed("roleId","role_id") \
                        .withColumnRenamed("languageRole","language_role") \
                        .withColumn("ingestion_date", current_timestamp()) \
                    .withColumn("environment", lit(v_environment)) \
                    .withColumn("file_date",lit(v_file_date))

#####Paso 3 - Escribir la salida en formato parquet

In [0]:
languages_roles_with_columns_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("movie_silver.languages_roles") 

In [0]:
%sql
SELECT * FROM movie_silver.languages_roles

role_id,language_role,ingestion_date,environment,file_date
1,Original,2026-11-04T21:11:53.169869Z,production,2024-12-16
2,Spoken,2026-11-04T21:11:53.169869Z,production,2024-12-16
